## Perform OCR and transform to images

In [2]:
# Import the libraries
from pdf2image import convert_from_path
import os

In [3]:
# Function to convert pdf into images and store the images into specified path
def pdf_to_images(pdf_path, output_folder):
  # Create the output folder if it doesn't exist
  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  # Convert PDF into images
  images = convert_from_path(pdf_path) # Convert each page of the PDF to an image
  image_paths = []

  # Save images and store their paths
  for i, image in enumerate(images):
    image_path = os.path.join(output_folder, f"page{i+1}.jpg") # Generate the image file path
    image.save(image_path, "JPEG") # Save the image as a JPEG file
    image_paths.append(image_path) # Append the image path to the list

  return image_paths # Return the list of image paths

In [4]:
# Define the path to the PDF and the output folder for images
pdf_path = "Things mother used to make.pdf"
output_folder = "images"

# Convert the PDF into images and store in the images folder
image_paths = pdf_to_images(pdf_path, output_folder)

In [5]:
# Import libraries
from dotenv import load_dotenv
from openai import OpenAI
import base64

load_dotenv()

client = OpenAI()
model = "gpt-5-nano"

In [6]:
# Read and encode one image
image_path = "images/page23.jpg" # Path to the image to be encoded

# Encode the image in base64 and decode to string
with open(image_path, "rb") as image_file:
  image_data = base64.b64encode(image_file.read()).decode('utf-8')
image_data

'...'

In [7]:
# Define the system prompt
system_prompt = """
Please analyze the content of this image and extract any related recipe information.
"""

In [8]:
# Call the OpenAI Responses API
response = client.responses.create(
    model=model,
    input=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "This is the image from the recipe page."},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{image_data}",
                    "detail": "low"
                }
            ]
        }
    ]
)

In [9]:
# Retrieve the content
gpt_response = response.output_text

In [10]:
from IPython.display import Markdown, display

# Define a function to display the response in Markdown
def get_gpt_response():
    return display(Markdown(gpt_response))

# Call the function
get_gpt_response()

Here are the recipes and ingredients shown on the page (“Things Mother Used to Make”):

Bannocks (Breads)
- Ingredients:
  - 1 cupful thick sour milk
  - 1/2 cupful sugar
  - 1 egg
  - 2 cupfuls flour
  - 1/2 cupful Indian meal
  - 1 teaspoonful soda
  - A pinch of salt
- Instructions:
  - Make the mixture stiff enough to drop from a spoon.
  - Drop the mixture, walnut-sized, into boiling fat.
  - Serve warm, with maple syrup.

Boston Brown Bread
- Ingredients:
  - 1 cupful rye meal
  - 1 cupful graham meal
  - 1/2 cupful flour
  - 1 cupful Indian meal
  - 1 cupful sweet milk
  - 1 cupful sour milk
  - 1 cupful molasses
  - 1/2 teaspoonful salt
  - 1 heaping teaspoonful of soda
- Instructions:
  - Stir the meals and salt together.
  - Beat the soda into the molasses until it foams.
  - Add sour milk and mix all together.
  - Pour into a greased tin pail (or brown-bread steamer).

Notes:
- The page is a vintage home-made bread/quick-bread guide with simple, rustic measurements (“cupful”).
- It suggests serving Bannocks warm with maple syrup.

In [11]:
# Define improved system prompt
system_prompt2 = """
Please analyze the content of this image and extract any relevant recipe information into structured components.
Specifically, extract the recipe title, list of ingredients, step-by-step instructions, cuisine type, dish type, and any relevant tags or metadata.
The output must be formatted in a way that is suitable for embedding in a Retrieval-Augmented Generation (RAG) system.
"""

In [12]:
# Call the API to extract the information
response = client.responses.create(
    model=model,
    input=[
        # System prompt
        {
            "role": "system",
            "content": system_prompt2
        },
        # User input (text + image)
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "This is the image from the recipe page"},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{image_data}",
                    "detail": "low"
                }
            ]
        }
    ]
)

In [13]:
# Print the info from the page with the improved prompt
get_gpt_response()

Here are the recipes and ingredients shown on the page (“Things Mother Used to Make”):

Bannocks (Breads)
- Ingredients:
  - 1 cupful thick sour milk
  - 1/2 cupful sugar
  - 1 egg
  - 2 cupfuls flour
  - 1/2 cupful Indian meal
  - 1 teaspoonful soda
  - A pinch of salt
- Instructions:
  - Make the mixture stiff enough to drop from a spoon.
  - Drop the mixture, walnut-sized, into boiling fat.
  - Serve warm, with maple syrup.

Boston Brown Bread
- Ingredients:
  - 1 cupful rye meal
  - 1 cupful graham meal
  - 1/2 cupful flour
  - 1 cupful Indian meal
  - 1 cupful sweet milk
  - 1 cupful sour milk
  - 1 cupful molasses
  - 1/2 teaspoonful salt
  - 1 heaping teaspoonful of soda
- Instructions:
  - Stir the meals and salt together.
  - Beat the soda into the molasses until it foams.
  - Add sour milk and mix all together.
  - Pour into a greased tin pail (or brown-bread steamer).

Notes:
- The page is a vintage home-made bread/quick-bread guide with simple, rustic measurements (“cupful”).
- It suggests serving Bannocks warm with maple syrup.

In [14]:
# Extract information about all of the images/recipes
extracted_recipes = []

for image_path in image_paths:
    print(f"Processing image {image_path}")

    # Read and encode the image
    with open(image_path, "rb") as image_file:
        image_data = base64.b64encode(image_file.read()).decode("utf-8")

    # Call the API to extract the information
    response = client.responses.create(
        model=model,
        input=[
            # Provide system prompt for guidance
            {
                "role": "system",
                "content": system_prompt2
            },
            # The user message contains both the text and image URL/path
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "This is the image from the recipe page"},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_data}",
                        "detail": "low"
                    }
                ]
            }
        ]
    )

    # Extract the content
    gpt_response = response.output_text

    # Store results
    extracted_recipes.append({
        "image_path": image_path,
        "recipe_info": gpt_response
    })

    print(f"Extracted information for {image_path}:\n{gpt_response}\n")

Processing image images/page1.jpg
Extracted information for images/page1.jpg:
{
  "recipe_title": "Things Mother Used to Make",
  "author": "Lydia Maria Gurney",
  "author_variants": ["Lydia Maria Gurney", "Lydia Maria Gurnex"],
  "ingredients": [],
  "steps": [],
  "cuisine_type": null,
  "dish_type": null,
  "tags": ["historical cookbook", "traditional home recipes", "mother's recipes"],
  "source": "Book cover image",
  "notes": "Only the cover is visible; no ingredients or step-by-step instructions present."
}

Processing image images/page2.jpg
Extracted information for images/page2.jpg:
{
  "recipe_title": null,
  "ingredients": [],
  "steps": [],
  "cuisine_type": null,
  "dish_type": null,
  "tags": [
    "title_page",
    "historical_document",
    "Cornell University",
    "School of Hotel Administration",
    "James B. Herndon Jr."
  ],
  "metadata": {
    "document_type": "title_page",
    "source": "image",
    "collection": "James B. Herndon Jr. Library",
    "institution"